# How the dataset is built — a walkthrough for one song

This notebook walks through the **entire preparation pipeline** on a single example,
so you can *listen* to both recordings and *see* the data at every stage. The batch
machinery that runs this over all songs lives in `align_lib.py` (audio alignment),
`vlm_lyrics.py` (lyric OCR), `map_lyrics_to_kr.py`, `align_kr_lyrics.py`, and
`build_bitext.py`.

**The task.** We have Mandarin **填詞** (fan lyric-rewrite) covers of K-pop songs and the
Korean originals. For training we want, per song:

1. a **time alignment** between the cover and the original (which moment in the cover
   corresponds to which moment in the original), and
2. the **Mandarin lyrics** with timestamps, projected onto the **Korean** timeline.

**Example song:** `054` — **Red Velvet – Peek-A-Boo (피카부)**, a genuine Mandarin-sung
**填詞** cover of a Korean original. What makes it a fun showcase: the author re-authored the
bright, playful song into a **dark horror-fairy-tale** rewrite (same melody, brand-new macabre
words — the video is even titled "惊声尖叫！黑暗童话?" / "Scream! A dark fairy tale?"). Every stage
below is meaningful because the cover really is sung in Mandarin.

In [ ]:
import os, sys, json, subprocess
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import librosa
from IPython.display import Audio, display
import matplotlib.font_manager as fm

# Use a CJK-capable font so Chinese characters can render in the matplotlib figures
for _fp in ["/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",
            "/usr/share/fonts/truetype/droid/DroidSansFallbackFull.ttf"]:
    if os.path.exists(_fp):
        fm.fontManager.addfont(_fp)
        plt.rcParams["font.sans-serif"] = [fm.FontProperties(fname=_fp).get_name()] + \
            plt.rcParams["font.sans-serif"]
        break
plt.rcParams["axes.unicode_minus"] = False

sys.path.insert(0, os.path.dirname(os.path.abspath("align_lib.py")))
import align_lib as A
from align_lib import SR, HOP, BASE

SONG_IDX = 54 # Red Velvet - Peek-A-Boo

song_pairs = {p[0]: p for p in A.load_pairs()}

# song_pairs is a dictionary mapping song index to a tuple of (index, name, cover_path, orig_path).
# Some indices are missing (hence the fact that this is a dictionary and not a list) because the Bilibili playlist
# includes multiple videos that differ from the original song. For example, several songs include not only the
# main cover but also an acapella version (纯人声) or an English version (英文版). For the purposes of this dataset,
# we mainly care about Mandarin Chinese and Korean pairs.

# song_pairs entry example:
# 54: (
#     54,
#     'Red Velvet - Peek-A-Boo',
#     '/path/to/bilibili_kpop_tianci/covers/054 - ....m4a',
#     '/path/to/bilibili_kpop_tianci/originals/Red Velvet - Peek-A-Boo.m4a'
# )
song_idx, song_name, cover_path, orig_path = song_pairs[SONG_IDX]

assert song_idx == SONG_IDX, f"song index mismatch: {song_idx} != {SONG_IDX}"

vpath = f"{BASE}/ocr/video/{SONG_IDX:03d}.mp4"     # cover video (subtitle frames read in section 5)
meta = {m["idx"]: m for m in json.load(open(f"{BASE}/covers_meta.json"))}
bv = meta[SONG_IDX]["bv"] # bv is the _B_ilibili _V_ideo ID

print("song        :", song_name)
print("cover  (ZH) :", os.path.basename(cover_path))
print("original(KR):", os.path.basename(orig_path))
print("bilibili    :", f"https://www.bilibili.com/video/{bv}")

## 1. The two source recordings

The **cover** is a Bilibili video with the Mandarin lyrics burned into the picture as
subtitles (we read those frames in section 5). The **original** is the Korean audio from
YouTube.

Listen to a short excerpt of each below. They are **not** time-aligned here — played at the
same offset they land on different musical moments, because the cover has an intro the
original lacks and is a re-sung Mandarin cover (different voice/language). Making them line
up is exactly what the rest of this notebook does — hear the **aligned** result in section 3.

In [ ]:
# A short excerpt of each recording. These are NOT aligned to each other (same offset =
# different musical moment); the aligned A/B is in section 3. Full audio is loaded here
# because later cells slice specific matched spans out of it.
cover_y, _ = librosa.load(cover_path, sr=SR, mono=True)
original_y, _ = librosa.load(orig_path,  sr=SR, mono=True)
start_s, duration_s = 60, 25

print(f"cover total {len(cover_y)/SR:.0f}s / original total {len(original_y)/SR:.0f}s — {duration_s}s excerpt from {start_s}s (unaligned)")
print("cover (Mandarin):");  display(Audio(cover_y[start_s*SR:(start_s+duration_s)*SR], rate=SR))
print("original (Korean):"); display(Audio(original_y[start_s*SR:(start_s+duration_s)*SR], rate=SR))

## 2. Feature extraction — HPSS-harmonic CQT chromagram

We can't compare the raw audio: the two recordings differ in voice, language, mix and key.
We need a representation that captures **the harmony/melody over time** while ignoring
timbre. That's the **chromagram**:

1. resample to 22.05 kHz mono,
2. **HPSS** — keep the *harmonic* component, drop percussion/transients,
3. **Constant-Q chroma** — fold the spectrum into the **12 pitch classes** (C, C#, …, B),
4. L2-normalize each frame.

Each column is one ~46 ms frame; each row is a pitch class. This is `align_lib.compute_chroma`.

In [ ]:
chroma_cover = A.chroma_cached(SONG_IDX, "cover", cover_path)   # 12 x T, cached under features/
chroma_original = A.chroma_cached(SONG_IDX, "orig",  orig_path)
print("cover chroma :", chroma_cover.shape, " (12 pitch classes x frames)")
print("orig  chroma :", chroma_original.shape)

fig, ax = plt.subplots(2, 1, figsize=(12, 6), gridspec_kw={"hspace": 0.5})
for axis, chroma, title in [(ax[0], chroma_cover, "Cover (Mandarin) — CQT chroma"),
                            (ax[1], chroma_original, "Original (Korean) — CQT chroma")]:
    img = librosa.display.specshow(chroma, y_axis="chroma", x_axis="time",
                                   sr=SR, hop_length=HOP, ax=axis, cmap="magma")
    axis.set(title=title, ylabel="pitch class")
fig.colorbar(img, ax=ax, format="%.1f", label="norm. energy")
plt.show()

## 3. Subsequence DTW alignment

With comparable features we run **subsequence Dynamic Time Warping** (`dtw-python`, cosine
distance, asymmetric step, `open_begin`/`open_end`). "Open begin/end" lets the shorter clip
match as a **contiguous chunk** of the longer one, so intros/outros that only one version has
are skipped cleanly. The output is a **warping path**: a dense list of
(cover_time ↔ original_time) correspondences.

In [ ]:
alignment = A.align(chroma_cover, chroma_original)   # does key-correction internally, then DTW
print(f"normalized DTW cost : {alignment['norm_distance']:.4f}   (lower = tighter harmonic match)")
print(f"key shift applied   : +{alignment['transpose_semitones']} semitones")
print(f"cover matched span  : {alignment['cover_span_sec'][0]:.0f}–{alignment['cover_span_sec'][1]:.0f}s")
print(f"orig  matched span  : {alignment['orig_span_sec'][0]:.0f}–{alignment['orig_span_sec'][1]:.0f}s")
print(f"path length         : {len(alignment['cover_sec'])} frame correspondences")

The warping path below maps **cover time → original time**. A straight diagonal means the
cover tracks the original at constant tempo (covers reuse the original backing track, so most
of the song is one clean diagonal). Flat/steep stretches mark passages one version holds
longer or skips.

In [ ]:
cover_sec, original_sec = alignment["cover_sec"], alignment["orig_sec"]
plt.figure(figsize=(6.5, 6.5))
plt.plot(cover_sec, original_sec, lw=1.6, color="crimson")
diag_lo = min(cover_sec.min(), original_sec.min())
diag_hi = min(cover_sec.max(), original_sec.max())
plt.plot([diag_lo, diag_hi], [diag_lo, diag_hi], ls="--", c="gray", lw=1, label="constant tempo (y=x)")
plt.xlabel("cover time (s)  — Mandarin"); plt.ylabel("original time (s)  — Korean")
plt.title(f"DTW warping path — {song_name}"); plt.legend(); plt.grid(alpha=.3)
plt.gca().set_aspect("equal"); plt.show()

### Hear that the alignment is right

Pick a moment in the cover, find the matched moment in the original via the warping path,
and play a few seconds of each. They should be the **same musical passage** — the Mandarin
cover vs the Korean original.

In [ ]:
def cover_to_original_time(cover_time):   # map a cover timestamp onto the original's, via the dense path
    return float(np.interp(cover_time, cover_sec, original_sec))

clip_s = 6.0
cover_time = float(cover_sec.min()) + 0.45 * (cover_sec.max() - cover_sec.min())   # ~45% into the matched span
original_time = cover_to_original_time(cover_time)
print(f"cover  {cover_time:5.1f}s  ↔  original {original_time:5.1f}s")
print("cover (Mandarin):");  display(Audio(cover_y[int(cover_time*SR):int((cover_time+clip_s)*SR)], rate=SR))
print("original (Korean):"); display(Audio(original_y[int(original_time*SR):int((original_time+clip_s)*SR)], rate=SR))

## 4. Matching time intervals (for training)

For training we don't want thousands of raw frame pairs — we want a handful of **intervals**.
`align_lib.segments` simplifies the warping path (Ramer–Douglas–Peucker) into segments, each
labelled `aligned` (same passage, tempo≈1), `cover_extra` (cover holds/adds material), or
`orig_extra` (original has material the cover skips/compresses).

In [ ]:
segment_list = A.segments(alignment["cover_sec"], alignment["orig_sec"], tol=1.5)
segments_df = pd.DataFrame(segment_list)

def to_mmss(seconds):
    return f"{int(seconds//60)}:{seconds%60:05.2f}"

display_df = segments_df.copy()
for col in ("kr_start", "kr_end", "zh_start", "zh_end"):
    display_df[col] = display_df[col].map(to_mmss)
display(display_df)

In [ ]:
# Visualize the segments on parallel timelines. The two rows are on independent clocks
# (Korean time vs Mandarin time), so a segment sits at DIFFERENT x-positions on each row.
# The shaded band links each segment across the rows: its slope shows the time offset
# (cover starts ~15s later) and its taper shows compression (orig_extra: Korean wider on
# the bottom than Mandarin on top). Read the bands, not vertical alignment.
fig, ax = plt.subplots(figsize=(12, 4))
kind_colors = {"aligned": "#4c9f70", "cover_extra": "#e0a458", "orig_extra": "#c0504d"}
KR_Y, ZH_Y, BAR_H = 0.0, 1.0, 0.16

for _, seg in segments_df.iterrows():
    color = kind_colors[seg.kind]
    # solid timeline bar on each row
    ax.add_patch(plt.Rectangle((seg.kr_start, KR_Y - BAR_H/2), seg.kr_end - seg.kr_start, BAR_H,
                               color=color, ec="white", lw=1))
    ax.add_patch(plt.Rectangle((seg.zh_start, ZH_Y - BAR_H/2), seg.zh_end - seg.zh_start, BAR_H,
                               color=color, ec="white", lw=1))
    # band linking the SAME segment across the two timelines
    ax.add_patch(plt.Polygon([(seg.kr_start, KR_Y + BAR_H/2), (seg.kr_end, KR_Y + BAR_H/2),
                              (seg.zh_end, ZH_Y - BAR_H/2), (seg.zh_start, ZH_Y - BAR_H/2)],
                             color=color, alpha=0.25))

ax.set_yticks([KR_Y, ZH_Y]); ax.set_yticklabels(["Korean\n(original)", "Mandarin\n(cover)"])
ax.set_ylim(-0.5, 1.5)
ax.set_xlim(-2, max(segments_df.kr_end.max(), segments_df.zh_end.max()) + 2)
ax.set_xlabel("time (s)")
ax.set_title("Matching segments — shaded band links the same passage across the two timelines")
legend_handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in kind_colors.values()]
ax.legend(legend_handles, kind_colors.keys(), ncol=3, loc="upper center", fontsize=8)
plt.tight_layout(); plt.show()

## 5. Reading the Mandarin lyrics (Qwen3-VL OCR)

The covers ship no lyric text — the words are **burned into the video** in a stylized font
that traditional OCR (EasyOCR, PaddleOCR) couldn't read. We sample frames at 2 fps and ask
**Qwen3-VL-8B** to read the subtitle on each frame (or say NONE), ignoring watermark/credits;
this is `vlm_lyrics.py`. Below are a few sampled frames with what the model read.

In [ ]:
# Pull the deduped lyric lines and the raw per-frame reads (both cached)
lyric_lines = [json.loads(l) for l in open(f"{BASE}/ocr/lyrics/{SONG_IDX:03d}.jsonl") if l.strip()]
raw_ocr = json.load(open(f"{BASE}/ocr/vlm_raw/{SONG_IDX:03d}.json"))
print(f"{len(raw_ocr['texts'])} frames read  ->  {len(lyric_lines)} deduped lyric lines")

# Show 3 frames from across the song, stacked full-width. We grab each frame at the line's
# MIDPOINT and prefer lines that stay on screen a beat (>=1s) so the frame matches the OCR
# text. We zoom the DISPLAY to the subtitle band (the OCR itself reads the full frame; the
# source is only 480p, so the crop is enlarged but soft). This cover puts the lyric in a box
# near the bottom; other covers place it elsewhere, which is why the OCR scans the whole frame.
durable = [l for l in lyric_lines if (l["end"] - l["start"]) >= 1.0] or lyric_lines
sample_lines = [durable[len(durable)//4], durable[len(durable)//2], durable[3*len(durable)//4]]
fig, ax = plt.subplots(3, 1, figsize=(9, 5.5))
for axis, line in zip(ax, sample_lines):
    mid = (line["start"] + line["end"]) / 2
    frame_path = f"/tmp/wt_{SONG_IDX}_{mid}.jpg"
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-ss", str(mid),
                    "-i", vpath, "-frames:v", "1", "-q:v", "2", frame_path], check=False)

    if os.path.exists(frame_path):
        frame = plt.imread(frame_path)
        h, w = frame.shape[:2]
        subtitle_region = frame[int(0.80 * h):int(0.97 * h), int(0.10 * w):int(0.90 * w)]  # bottom lyric box
        axis.imshow(subtitle_region, interpolation="lanczos")

    axis.axis("off")
    axis.set_title(f'{mid:.1f}s   —   OCR read: {line["text"]}', fontsize=11)
plt.tight_layout(); plt.show()

In [ ]:
# consecutive identical/near-identical frames are merged into one timed line
pd.DataFrame(lyric_lines[:12])[["start", "end", "text"]]

**Why per-frame stills and not native video mode?** Feeding the clip to the model as a
video forces a small frame budget (≈1 frame every few seconds for a 3-min song) and
downsamples the picture, so it *misses* fast-changing subtitle lines. On song 009 we measured
per-frame recall **54 lines vs. 33** in video mode — reading burned-in subtitles is a
still-image OCR task, and audio adds nothing (the timing comes from the audio alignment above).

## 6. The finished product — Mandarin lyrics on the Korean timeline

Finally we push each lyric line's cover timestamps **through the warping path** to get its
Korean-original timestamps (`map_lyrics_to_kr.py`, using `np.interp` over the dense path).
That yields the training-ready table: every Mandarin line with both its cover time and the
corresponding time in the Korean original.

In [ ]:
lyrics_aligned = pd.read_csv(f"{BASE}/ocr/lyrics_aligned/{SONG_IDX:03d}.csv")
print(f"{len(lyrics_aligned)} lines mapped onto the Korean timeline")
lyrics_aligned[["line", "zh_text", "zh_start", "zh_end", "kr_start", "kr_end"]].head(15)

In [ ]:
# Hear one line both ways: the Mandarin cover slice and the mapped Korean original slice
line_row = lyrics_aligned.iloc[len(lyrics_aligned)//2]
print(f'line {int(line_row.line)}: "{line_row.zh_text}"')
print(f"cover {line_row.zh_start:.1f}-{line_row.zh_end:.1f}s   ↔   original {line_row.kr_start:.1f}-{line_row.kr_end:.1f}s")
print("Mandarin (cover):");  display(Audio(cover_y[int(line_row.zh_start*SR):int(line_row.zh_end*SR)+SR], rate=SR))
print("Korean (original):"); display(Audio(original_y[int(line_row.kr_start*SR):int(line_row.kr_end*SR)+SR], rate=SR))

## 7. One-to-one Korean ↔ Mandarin lyric pairs

Section 6 placed the Mandarin lines on the Korean timeline, but we still don't have the
**Korean words**. To pair the two *lyrics*, we take the official Korean lyrics as text and
**force-align** them to the Korean audio (`align_kr_lyrics.py`, a multilingual CTC aligner)
to get each Korean line's timing. Both sides now live on the Korean timeline, so we **join
by time overlap** — each Mandarin cover line is attached to the Korean line it overlaps most
(`build_bitext.py`). Each pair is then enriched with:

- **Korean** (Hangul; the song mixes English, kept as-is) + **Revised Romanization**
- **Simplified** Chinese (the OCR'd cover) + **Traditional** Chinese (OpenCC, Taiwan variant)
- **English** translation (parallel `.en.txt`)

The pairing is by **song position, not literal meaning** — a 填詞 cover rewrites each line to
fit the melody. Here that's vivid: "맞아 난 좀 기분파" ("yeah, I go by my mood") ↔ "迈向永眠的步伐"
("steps toward eternal sleep") — same moment and melody, but the author flipped Peek-A-Boo's
bright, flirty line into something macabre. That cover-vs-original contrast is exactly the
training signal.

In [ ]:
# Cached by build_bitext.py: one row per Korean line, with the cover line(s) that overlap it,
# plus Traditional (OpenCC), Revised Romanization, and the English translation.
bitext = pd.read_csv(f"{BASE}/bitext/{SONG_IDX:03d}.csv")
print(f"{len(bitext)} one-to-one Korean <-> Mandarin lyric pairs")
bitext[["korean", "korean_rr", "zh_hans", "zh_hant", "english"]].head(12)

In [ ]:
# Hear a matched pair: the Korean ORIGINAL line vs the Mandarin COVER line at the same moment.
# Pick a Korean (Hangul) line that has a cover match, ~1/3 of the way in.
cand = bitext[bitext.zh_start.notna() & bitext.korean.str.contains(r"[가-힣]")]
pair = cand.iloc[len(cand) // 3]
print(f"Korean   : {pair.korean}   ({pair.korean_rr})")
print(f"English  : {pair.english}")
print(f"Mandarin : {pair.zh_hans}   /  {pair.zh_hant}")
print(f"Korean {pair.kr_start:.1f}-{pair.kr_end:.1f}s   |   cover {pair.zh_start:.1f}-{pair.zh_end:.1f}s")
print("Korean (original):");  display(Audio(original_y[int(pair.kr_start*SR):int(pair.kr_end*SR)], rate=SR))
print("Mandarin (cover):");   display(Audio(cover_y[int(pair.zh_start*SR):int(pair.zh_end*SR)], rate=SR))

**That's the whole pipeline for one song:**

1. two recordings →
2. HPSS-harmonic CQT chromagrams →
3. subsequence DTW warping path →
4. matching time intervals →
5. Qwen3-VL lyric OCR →
6. Mandarin lyrics projected onto the Korean timeline →
7. force-align the Korean lyrics and join → one-to-one Korean ↔ Mandarin pairs (Hangul + Revised Romanization, Simplified + Traditional Chinese, English).

(The aligner also checks for a **key shift** before DTW; genuine 中文版 covers reuse the
original backing track, so it's always 0 semitones here — that's why there's no key-correction
section. It only kicks in on covers re-sung in a different key.)

Run over all Chinese-cover pairs, this produces `lyrics_aligned.csv` (Mandarin lines with
Korean-timeline timestamps), the per-song `segments/` intervals, `ALIGNMENT_REPORT.md`, and
`bitext.csv` (the enriched Korean↔Mandarin lyric pairs).